In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
ireland_2024= spark.read.parquet("s3a://bronze/IRELAND/2024_BRONZE/")

2026-07-13 23:54:29 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/13 23:55:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-07-13 23:56:46 | INFO | lakehouse.__main__ | Spark session created successfully


26/07/13 23:57:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
ireland_24=ireland_2024
ireland_24

DataFrame[information_for_the_purposes_of_transparency_pursuant_to_article_58: string, unnamed:_1: string, unnamed:_2: string, unnamed:_3: string, unnamed:_4: string, unnamed:_5: string, unnamed:_6: string, unnamed:_7: string, unnamed:_8: string, unnamed:_9: string, unnamed:_10: string, unnamed:_11: string, unnamed:_12: string, unnamed:_13: string, unnamed:_14: string, unnamed:_15: string, source_country: string, source_year: string, ingested_at: timestamp]

In [ ]:
ireland_24.rdd.getNumPartitions()

In [5]:
ireland_24.printSchema()

root
 |-- information_for_the_purposes_of_transparency_pursuant_to_article_58: string (nullable = true)
 |-- unnamed:_1: string (nullable = true)
 |-- unnamed:_2: string (nullable = true)
 |-- unnamed:_3: string (nullable = true)
 |-- unnamed:_4: string (nullable = true)
 |-- unnamed:_5: string (nullable = true)
 |-- unnamed:_6: string (nullable = true)
 |-- unnamed:_7: string (nullable = true)
 |-- unnamed:_8: string (nullable = true)
 |-- unnamed:_9: string (nullable = true)
 |-- unnamed:_10: string (nullable = true)
 |-- unnamed:_11: string (nullable = true)
 |-- unnamed:_12: string (nullable = true)
 |-- unnamed:_13: string (nullable = true)
 |-- unnamed:_14: string (nullable = true)
 |-- unnamed:_15: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_year: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [6]:
for i , col in enumerate(ireland_24.columns, 1):
    print(f"{i:3d}| {col}")

  1| information_for_the_purposes_of_transparency_pursuant_to_article_58
  2| unnamed:_1
  3| unnamed:_2
  4| unnamed:_3
  5| unnamed:_4
  6| unnamed:_5
  7| unnamed:_6
  8| unnamed:_7
  9| unnamed:_8
 10| unnamed:_9
 11| unnamed:_10
 12| unnamed:_11
 13| unnamed:_12
 14| unnamed:_13
 15| unnamed:_14
 16| unnamed:_15
 17| source_country
 18| source_year
 19| ingested_at


In [7]:
ireland_24.select(
    "information_for_the_purposes_of_transparency_pursuant_to_article_58",
    "unnamed:_1",
    "unnamed:_2",
    "unnamed:_3"
    
).limit(10).toPandas()


,information_for_the_purposes_of_transparency_pursuant_to_article_58,unnamed:_1,unnamed:_2,unnamed:_3
0,None,None,None,None
1,Name of the beneficiary/Legal entity/association,Surname of beneficiary,"If belonging to a group, name of the parent en...",Municipality
2,None,None,None,None
3,MICHAEL CLANCY,CLANCY,None,LONGFORD
4,MICHAEL CLANCY,CLANCY,None,LONGFORD
5,MICHAEL CLANCY,CLANCY,None,LONGFORD
6,MICHAEL CLANCY,CLANCY,None,LONGFORD
7,MICHAEL CLANCY,CLANCY,None,LONGFORD
8,MICHAEL CLANCY,CLANCY,None,LONGFORD
9,None,None,None,None


In [5]:
#since we know that the column name is in th second row , we are extracting it and creating a new dataframe
new_column_names= []

sample_rows= ireland_24.take(2)[1]

legit_columns= ["source_country", "source_year", "ingested_at"]
# Conveert the raw values into a clean python list

for i , old_name in enumerate(ireland_24.columns):
    if old_name in legit_columns:
        new_column_names.append(old_name)
    else:
        new_column_names.append(str(sample_rows[i]))
# map the old column names to the new one
rename_map= dict(zip(ireland_24.columns, new_column_names))
df_with_headers= ireland_24.withColumnsRenamed(rename_map)

sample_new_col= [col for col in new_column_names if col not in legit_columns][0]
ireland_24_new=df_with_headers.filter(F.col(sample_new_col) != sample_new_col) 
ireland_24_new.show(5, truncate=False)


+------------------------------------------------+----------------------+---------------------------------------------------------------------------------------+------------+--------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+-------------------+------------------------------+-----------------------------------------+-------------------------------+------------------------------------------+--------------------------------------+------------------------------------------------+--------------------------------------+-------------------------------------------+--------------+------

In [16]:
ireland_24_new.show(5)

+------------------------------------------------+----------------------+---------------------------------------------------------------------------------------+------------+--------------------------------------------------------------+---------------------+-------------------+-------------------+------------------------------+-----------------------------------------+-------------------------------+------------------------------------------+--------------------------------------+------------------------------------------------+--------------------------------------+-------------------------------------------+--------------+-----------+--------------------+
|Name of the beneficiary/Legal entity/association|Surname of beneficiary|If belonging to a group, name of the parent entity and VAT or Tax identification number|Municipality|Code of measure type of intervention/sector as set in Annex IX|Specific objective Â¹|      Start date Â²|        End date Â³|Amount by operation under EAGF|Tot

In [6]:
# let's know the number of non null or nan values present in each column

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []


for col_name, col_type in ireland_24_new.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()

    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))
    
    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
    
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = ireland_24_new.select(count_expressions).first()

print(counts_row)


Row(Name of the beneficiary/Legal entity/association=571003, Surname of beneficiary=480559, If belonging to a group, name of the parent entity and VAT or Tax identification number=4216, Municipality=570172, Code of measure type of intervention/sector as set in Annex IX=570172, Specific objective Â¹=544317, Start date Â²=544179, End date Â³=544248, Amount by operation under EAGF=377289, Total of EAGF amount for that beneficiary=0, Amount by operation under EAFRD=192814, Total of EAFRD amount for that beneficiary=0, Amount by operation under co-financing=178964, Total of co-financed amount for that beneficiary=0, Total of EAFRD and co-financed amounts=0, Total of the EU amount for that beneficiary=0, source_country=571003, source_year=571003, ingested_at=571003)


In [ ]:
# Count total rows in the DataFrame
total_rows = ireland_24_new.count()

# Print header
print(f'{"Column Name": <65} | {"Missing Percentage"}')
print("-" * 85)

# Calculate and print missing percentage for each column
for column, valid_count in counts_row.asDict().items():
    missing_percentage = ((total_rows - valid_count) / total_rows) * 100
    print(f"{column: <65} | {missing_percentage: >10.3f}%")


In [8]:
# Get column data types as a dictionary
col_types = dict(ireland_24_new.dtypes)

# Count total rows in the DataFrame
total_rows = ireland_24_new.count()

# Initialize list to hold stats
stats_list = []

# Extract stats from data
for col_name, valid_count in counts_row.asDict().items():
    missing_count = total_rows - valid_count
    missing_percentage = round((missing_count / total_rows) * 100, 2)

    stats_list.append({
        'column_name': col_name,
        'missing_count': missing_count,
        'missing_percentage': missing_percentage,
        'Non-Null count': valid_count,
        'dtype': col_types.get(col_name)
    })
# Sort missing_percentage in ascending order
missing_ireland_stats_2024 = sorted(
    stats_list,
    key=lambda x: x['missing_percentage']
)

# Display the sorted list
missing_ireland_stats_2024

# Convert to DataFrame (optional, for display)
import pandas as pd
stats_df = pd.DataFrame(stats_list)

# Show nicely formatted table
print(stats_df)




                                          column_name  missing_count  \
0    Name of the beneficiary/Legal entity/association              0   
1                              Surname of beneficiary          90444   
2   If belonging to a group, name of the parent en...         566787   
3                                        Municipality            831   
4   Code of measure type of intervention/sector as...            831   
5                               Specific objective Â¹          26686   
6                                       Start date Â²          26824   
7                                         End date Â³          26755   
8                      Amount by operation under EAGF         193714   
9           Total of EAGF amount for that beneficiary         571003   
10                    Amount by operation under EAFRD         378189   
11         Total of EAFRD amount for that beneficiary         571003   
12             Amount by operation under co-financing         39

In [ ]:
ireland_24_cleaned= ireland_24_new.select(
    F.col("Name of the beneficiary/Legal entity/association").alias("beneficiary"),
    F.col("Municipality").alias("municipality"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    
    F.col("Code of measure type of intervention/sector as set in Annex IX").alias("intervention_code"),
    F.col("Amount by operation under EAGF").cast(DoubleType()).alias("total_eagf_income_support"),
    F.col("Amount by operation under EAFRD").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.col("Amount by operation under co-financing").cast(DoubleType()).alias("national_cofunding_amount")
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
ireland24_cleaned= ireland_24_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [ ]:
ireland24_cleaned..show(5, truncate=False)

In [6]:
# change the data type of columns that handles subsidy payments from string to double 
convers_exprs =[]

for col_name in ireland_24_new.columns:
    #get current data type
    current_type= ireland_24_new.schema[col_name].dataType
    if isinstance(current_type, StringType) and "amount" in col_name.lower():
        #only cast if it is currently a string
        convers_exprs.append(F.col(col_name).cast(DoubleType()).alias(col_name))
    else:
        convers_exprs.append(F.col(col_name))

ireland24_cleaned= ireland_24_new.select(convers_exprs)


In [11]:
# we know we have over 10,000 distinct surnames 
# But we need to verify if there is no value assigned for the corresponding row value for name of beneficiary

#1. let's define a condition that makes a name "empty" and a surname "valid"

name_is_empty= (
    F.col("Name of the beneficiary/Legal entity/association").isNull() |
    F.col("Name of the beneficiary/Legal entity/association").isin("None", "N/A", "n/a", "")
    )

surname_is_valid= (
    F.col("Surname of beneficiary").isNotNull() &
    ~F.col("Surname of beneficiary").isin("None", "N/A", "n/a", "")
    )

# combine them : Name is broken but surname has a real value
name_shifted=  name_is_empty & surname_is_valid

# count affected rows
affected_names_count= ireland24_cleaned.filter(name_shifted).count()
print(f"Rows where name is empty but surname isn't: {affected_names_count}")


Rows where name is empty but surname isn't: 0


In [12]:
ireland24_cleaned.select(
    "Start date Â²",
    "End date Â³"
    
).distinct().limit(5).show()


+-------------------+-------------------+
|      Start date Â²|        End date Â³|
+-------------------+-------------------+
|2023-12-15 00:00:00|2024-06-21 00:00:00|
|2023-10-24 00:00:00|2023-12-04 00:00:00|
|2023-10-19 00:00:00|2024-09-18 00:00:00|
|2023-12-18 00:00:00|2023-12-18 00:00:00|
|2023-12-04 00:00:00|2024-06-17 00:00:00|
+-------------------+-------------------+



In [13]:
ireland24_cleaned.select(
    "If belonging to a group, name of the parent entity and VAT or Tax identification number",
    "Specific objective Â¹"
).distinct().limit(5).show()


+---------------------------------------------------------------------------------------+---------------------+
|If belonging to a group, name of the parent entity and VAT or Tax identification number|Specific objective Â¹|
+---------------------------------------------------------------------------------------+---------------------+
|                                                                             IE9760771A| The basic income ...|
|                                                                             IE0650202O| The aim of the ai...|
|                                                                            IE4093945AH| Eco-schemes are a...|
|                                                                            IE4093945AH| The complementary...|
|                                                                            IE3293031FH| The basic income ...|
+---------------------------------------------------------------------------------------+---------------

In [ ]:
ireland24_cleaned.printSchema()

In [7]:
# note we are removing these two columns from the aligned datasets since they cannot be identified 
# [Total of EAFRD and co-financed amounts, Total of the EU amount for that beneficiary]
beneficiary_key = "Name of the beneficiary/Legal entity/association"

# Step 1: Perform the GroupBy and correctly aggregate using F.sum and F.col
ireland24_merged = ireland24_cleaned.groupBy(
    beneficiary_key, "Code of measure type of intervention/sector as set in Annex IX"
).agg(
    # EAGF merge (Summing to capture all records per beneficiary)
    F.sum(F.coalesce(F.col("Amount by operation under EAGF"))).alias("total_eagf_income_support"),
    
    # EAFRD merge
    F.sum(F.coalesce(F.col("Amount by operation under EAFRD"))).alias("total_eafrd_income_support"),

    # Co-Financing Merge
    F.sum(F.coalesce(F.col("Amount by operation under co-financing"))).alias("total_irish_co_funding"),
    
    # Metadata tracking text columns
    F.first("source_country", ignorenulls=True).alias("country"),
    F.first("source_year", ignorenulls=True).alias("year"),
    F.first("Municipality").alias("municipality"),
    F.first("If belonging to a group, name of the parent entity and VAT or Tax identification number", ignorenulls=True).alias("parent_entity_vat")
)

# Step 2: Add your calculated grand total column directly to the merged DataFrame
# Note: True EU Payout = EAGF + EAFRD
ireland24_merged = ireland24_merged.withColumn(
    "grand_total_eu_payout",
    F.coalesce(F.col("total_eagf_income_support"), F.lit(0.0)) + F.coalesce(F.col("total_eafrd_income_support"), F.lit(0.0) + F.coalesce(F.col("total_irish_co_funding"), F.lit(0.0))
)

# Step 3: Now define your complete numeric list, including the new calculated column
final_numeric_columns = [
    "total_eagf_income_support",
    "total_eafrd_income_support",
    "total_irish_co_funding",
    "grand_total_eu_payout"
]

# Step 4: Finalize the revamped DataFrame by filling nulls on the fully computed dataset
ireland24_revamped = ireland24_merged.na.fill(0.0, subset=final_numeric_columns)

# Verify the results
ireland24_revamped.limit(5).show(truncate=False)


+------------------------------------------------+--------------------------------------------------------------+-------------------------+--------------------------+----------------------+-------+----+------------+-----------------+---------------------+
|Name of the beneficiary/Legal entity/association|Code of measure type of intervention/sector as set in Annex IX|total_eagf_income_support|total_eafrd_income_support|total_irish_co_funding|country|year|municipality|parent_entity_vat|grand_total_eu_payout|
+------------------------------------------------+--------------------------------------------------------------+-------------------------+--------------------------+----------------------+-------+----+------------+-----------------+---------------------+
|(b)                                             |NULL                                                          |0.0                      |0.0                       |0.0                   |IRELAND|2024|NULL        |NULL             

In [15]:
ireland24_revamped.printSchema()

root
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- Code of measure type of intervention/sector as set in Annex IX: string (nullable = true)
 |-- total_eagf_income_support: double (nullable = false)
 |-- total_eafrd_income_support: double (nullable = false)
 |-- total_irish_co_funding: double (nullable = false)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- parent_entity_vat: string (nullable = true)
 |-- grand_total_eu_payout: double (nullable = false)



In [ ]:
distinct_exprs= [F.approx_count_distinct(F.col(c),rsd=0.05).alias(c) for c in ireland24_revamped.columns]
distinct_rows= ireland24_revamped.select(distinct_exprs).first()
count_dict= distinct_rows.asDict()

for col, count in count_dict.items():
    print(f"{col:<35} | Distinct Values {count:,}")


In [16]:
# 1. Warm up and lock in the cache safely
# ireland24_revamped.cache()
total_rows = ireland24_revamped.count()
print(f"Data cached successfully. Total rows: {total_rows:,}\n")

# 2. Settings for batching
all_columns = ireland24_revamped.columns
batch_size = 4  # Keeps memory footprint completely stable
final_counts = {}

print("Computing distinct counts in stable batches...")

# 3. Process columns in chunks to prevent memory spikes
for i in range(0, len(all_columns), batch_size):
    chunk = all_columns[i : i + batch_size]

    # Run the aggregation ONLY on this small batch of columns
    batch_exprs = [
        F.approx_count_distinct(c, rsd=0.05).alias(c) for c in chunk
    ]
    batch_result = ireland24_revamped.agg(*batch_exprs).first()

    # Save the results into our dictionary
    for c in chunk:
        final_counts[c] = batch_result[c]

# 4. Print your final clean output
print(f"\n{'Column Name':<55} | {'Approx Distinct Count'}")
print("-" * 80)
for col_name, count in final_counts.items():
    print(f"{col_name:<55} | {count:,}")


Data cached successfully. Total rows: 432,529

Computing distinct counts in stable batches...



Column Name                                             | Approx Distinct Count
--------------------------------------------------------------------------------
Name of the beneficiary/Legal entity/association        | 88,727
Code of measure type of intervention/sector as set in Annex IX | 30
total_eagf_income_support                               | 130,387
total_eafrd_income_support                              | 99,005
total_irish_co_funding                                  | 84,993
country                                                 | 1
year                                                    | 1
municipality                                            | 28
parent_entity_vat                                       | 1,034
grand_total_eu_payout                                   | 215,567


In [8]:
# Define your key columns
beneficiary_key = 'Name of the beneficiary/Legal entity/association'
scheme_key = 'Code of measure type of intervention/sector as set in Annex IX'
municipality_key = 'municipality'

# Filter out null beneficiaries
ireland24_core = ireland24_revamped.filter(F.col(beneficiary_key).isNotNull())

# Step 1: Clean and fetch distinct schemes using PySpark natively
distinct_schemes_df = ireland24_revamped \
    .select(scheme_key) \
    .filter(~F.col(scheme_key).isin(r"N\a", r"n\a", "")) \
    .dropna() \
    .distinct()

# Step 2: Bring only the final, clean list over to the driver
distinct_schemes = [row[0] for row in distinct_schemes_df.collect()]

print(f" Distinct schemes in intervention_sector code are :{distinct_schemes}")


 Distinct schemes in intervention_sector code are :['I.1', 'I.2', 'V.1', 'Leader Quality of Life and Diversification', 'Investments in Physical Assets', 'Beef & Veal Payments', 'V.2', 'I.3', 'V.8', 'Agri Environment Climate', 'Direct Payments', 'V.7', 'III.2', 'V.4', 'Specific Support Direct Aid', 'I.6', 'Animal Welfare', 'Organic Farming', 'Areas facing Natural/Specific Constraints', 'Co-Operation', 'Basic Services & Village Renewal', 'School Fruit Veg & Milk Scheme', 'Advisory Service, Farm Management & Relief', 'I.4', 'Natura 2000 & Water Framework Directive', 'III.1', 'Aid to Producer Organisations', 'Technical Assistance EAFRD', 'Food Promotion', 'Knowledge Transfer & Information Actions']


#### Map intervention sector code to real and self-explanatory schemes based on the Commission Implementing Regulation EU

In [9]:
Scheme_Registry = {
    # Pillar 1 Direct Support
    "I.1": ("direct_payment_biss", "Basic Income Support for Sustainability"),
    "Direct Payments": ("direct_payment_biss", "Basic Income Support for Sustainability"),
    "I.2": ("redistributive_support", "Complementary Redistributive Income Support"),
    "I.3": ("young_farmer_income", "Complementary Income Support for Young Farmers"),
    "I.4": ("eco_schemes", "Echo Schemes"),
    
    # Unified livestock & Animal husbandry
    "I.6": ("livestock_welfare_and_support", "Livestock Production and Animal Welfare"),
    "Beef & Veal Payments": ("livestock_welfare_and_support", "Livestock Production and Animal Welfare"),    
    "Specific Support Direct Aid": ("livestock_welfare_and_support", "Livestock Production and Animal Welfare"),
    "Animal Welfare": ("livestock_welfare_and_support", "Livestock Production and Animal Welfare"),

    # Sectoral / Market
    "School Fruit Veg & Milk Scheme": ("school_nutrition_program", "School Scheme"),
    "III.1": ("producer_org_aid", "Fruits and Vegetable Support"),
    "III.2": ("apiculture_support", "Apiculture Sector Support"),
    "Food Promotion": ("market_food_promotion", "Food Promotion"),

    # Rural Development & Environment
    "Agri Environment Climate": ("agri_environmental_commitments", "Environments and Climate Commitments"),
    "V.1": ("agri_environmental_commitments", "Environments and Climate Commitments"),
    "Organic Farming": ("agri_environmental_commitments", "Environments and Climate Commitments"),
    "V.2": ("natural_constraints_anc", "Areas Facing Natural Constraints"),
    "Areas facing Natural/Specific Constraints": ("natural_constraints_anc", "Areas Facing Natural Constraints"),
    "Natura 2000 & Water Framework Directive": ("natura_2000_disadvantages", "Natura 2000 and Water Framework Directive"),
    "V.7": ("physical_capital_investments", "Investments in Physical Assets"),
    "Investments in Physical Assets": ("physical_capital_investments", "Investments in Physical  Assets"),
    "V.8":("rural_business_startup", "Rural Startup and Young Farmer Setup"),
    "Leader Quality of Life and Diversification": ("leader_community_dev", "LEADER Local Action"),
    "Co-Operation": ("cooperation_innovation", "Co-Operation Projects"),
    "Basic Services & Village Renewal": ("village_renewal_infra", "BasicServices and Village Renewal"),
    "Advisory Service, Farm Management & Relief": ("farm_advisory_services", "Advisory Services"),
    "Knowledge Transfer & Information Actions": ("knowledge_transfer_training", "Knowledge Transfer Actions"),
    "Technical Assistance EAFRD": ("technical_assistance_eafrd", "Technical Assistance")
}

In [10]:
## horizontal pivot with expanded metric anchors
beneficiary_key = 'Name of the beneficiary/Legal entity/association'
intervention_key = 'Code of measure type of intervention/sector as set in Annex IX'
municipality_key = 'municipality'
vat_key = 'parent_entity_vat'

anchors = [vat_key, beneficiary_key, municipality_key, "country", "year"]

# Drop dead database rows using grand_total_payout
ireland24_filtered = ireland24_revamped.filter(
    ~(
        ((F.col(beneficiary_key).isNull()) | (F.trim(F.col(beneficiary_key)) == "")) &
        ((F.col(vat_key).isNull()) | (F.trim(F.col(vat_key)) == "")) &
        (F.col("grand_total_eu_payout") == 0.0)
    )
)


In [ ]:
total_rows = ireland24_filtered.count()
print(f"Data cached successfully. Total rows: {total_rows:,}\n")

# 2. Settings for batching
all_columns = ireland24_filtered.columns
batch_size = 4  # Keeps memory footprint completely stable
final_counts = {}

print("Computing distinct counts in stable batches...")

# 3. Process columns in chunks to prevent memory spikes
for i in range(0, len(all_columns), batch_size):
    chunk = all_columns[i : i + batch_size]

    # Run the aggregation ONLY on this small batch of columns
    batch_exprs = [
        F.approx_count_distinct(c, rsd=0.05).alias(c) for c in chunk
    ]
    batch_result = ireland24_filtered.agg(*batch_exprs).first()

    # Save the results into our dictionary
    for c in chunk:
        final_counts[c] = batch_result[c]

# 4. Print your final clean output
print(f"\n{'Column Name':<55} | {'Approx Distinct Count'}")
print("-" * 80)
for col_name, count in final_counts.items():
    print(f"{col_name:<55} | {count:,}")


In [11]:
# Mint fingerPrint using grand_total_eu_payout
anon_fingerprint = F.substring(
    F.md5(F.concat_ws("_", "grand_total_eu_payout", intervention_key)), 1, 7
)

ireland24_identifiable = ireland24_filtered \
    .withColumn(
        vat_key,
        F.when((F.col(vat_key).isNull()) | (F.trim(F.col(vat_key)) == ""), 
               F.concat(F.lit("ANON_VAT_"), F.col("country"), F.lit("_"), anon_fingerprint))
         .otherwise(F.col(vat_key))
    ) \
    .withColumn(
        beneficiary_key,
        F.when((F.col(beneficiary_key).isNull()) | (F.trim(F.col(beneficiary_key)) == ""), 
               F.concat(F.lit("ANON_NAME_"), F.col("country"), F.lit("_"), anon_fingerprint))
         .otherwise(F.col(beneficiary_key))
    )


## Macro pillar aggregration

In [12]:
co_funding_key="total_irish_co_funding"

has_co_funding= co_funding_key in ireland24_identifiable.columns

macro_expressions = [
    F.sum("total_eagf_income_support").alias("total_eagf_income_support"),
    F.sum("total_eafrd_income_support").alias("total_eafrd_income_support")
]

if has_co_funding:
    macro_expressions.append(F.sum(co_funding_key).alias("national_co_funding"))
else:
    macro_expressions.append(F.lit(0.0).alias("national_co_funding"))

ireland24_macros= ireland24_identifiable.groupBy(anchors).agg(*macro_expressions) 



# Macro scheme  horizontal pivot
mapping_expr = F.when(F.col(intervention_key).isNull(), "other_rural_development")
for raw_val, (token, _) in Scheme_Registry.items():
    mapping_expr = mapping_expr.when(F.col(intervention_key).contains(raw_val), token)
mapping_expr = mapping_expr.otherwise("other_rural_development")

# Add mapped scheme token column
ireland24_prepared = ireland24_identifiable.withColumn("scheme_token", mapping_expr)

# Example pivot (expand as needed)
ireland24_pivoted_micros= ireland24_prepared \
       .groupBy(anchors) \
       .pivot("scheme_token") \
       .agg(F.sum("grand_total_eu_payout")) \
       .na.fill(0.0)

#Merge
ireland24_master= ireland24_macros.join(ireland24_pivoted_micros, on=anchors,  how="inner")
ireland24_master.printSchema()

root
 |-- parent_entity_vat: string (nullable = true)
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- total_eagf_income_support: double (nullable = true)
 |-- total_eafrd_income_support: double (nullable = true)
 |-- national_co_funding: double (nullable = true)
 |-- agri_environmental_commitments: double (nullable = false)
 |-- cooperation_innovation: double (nullable = false)
 |-- direct_payment_biss: double (nullable = false)
 |-- eco_schemes: double (nullable = false)
 |-- farm_advisory_services: double (nullable = false)
 |-- knowledge_transfer_training: double (nullable = false)
 |-- leader_community_dev: double (nullable = false)
 |-- livestock_welfare_and_support: double (nullable = false)
 |-- market_food_promotion: double (nullable = false)
 |-- natura_2000_disadvantages: double (nullable = false)
 |-- natural_constraints_

In [13]:
macro_pillars= ["total_eagf_income_support","total_eafrd_income_support", "national_co_funding"]

# Seperate our micro-scheme columns from our anchors and macro keys
pivoted_scheme_columns= [ c for c in ireland24_master.columns if c not in anchors and c not in macro_pillars]

agg_exprs = []


#Drop flat lines of zeros
# we loop through only the micro-scheme columns , if min==max. It means every single row is identical (like 0.0) and signalling a dead column
for col_name in pivoted_scheme_columns:
    agg_exprs.append(F.min(col_name).alias(f"min_{col_name}"))
    agg_exprs.append(F.max(col_name).alias(f"max_{col_name}"))

# executes exactly one action across cluster instead of several separate loops
stats_row= ireland24_master.agg(*agg_exprs).first()

active_columns = []
if stats_row:
    stats_dict= stats_row.asDict()
    for col_name in pivoted_scheme_columns:
    
        if stats_dict[f"min_{col_name}"] != stats_dict[f"max_{col_name}"]:
            active_columns.append(col_name)
        else:
            print(f" Prunning zero-varaince column: {col_name}")

print(f"Phase 4 Completed, Found {len(active_columns)} micro-schemes remaining for the analysis:{active_columns}")

26/07/14 00:05:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Phase 4 Completed, Found 19 micro-schemes remaining for the analysis:['agri_environmental_commitments', 'cooperation_innovation', 'direct_payment_biss', 'eco_schemes', 'farm_advisory_services', 'knowledge_transfer_training', 'leader_community_dev', 'livestock_welfare_and_support', 'market_food_promotion', 'natura_2000_disadvantages', 'natural_constraints_anc', 'other_rural_development', 'physical_capital_investments', 'redistributive_support', 'rural_business_startup', 'school_nutrition_program', 'technical_assistance_eafrd', 'village_renewal_infra', 'young_farmer_income']


### Extraction of Domain-Driven Data Marts 

In [14]:
# The absolute minimum columns  required from ireland24_master to compute totals
#macro_pillars= ["total_eagf_income_support","total_eafrd_income_support", "national_co_funding"]

required_base_inputs= anchors + macro_pillars

master_tools = ["grand_total_public_payout", "grand_total_eu_payout"]

# Establish the universal tracking baseline that accompanies every sub-schema
mart_baseline= anchors + master_tools

# Schema 1: Direct support mart (Pillar 1)

# Tracks foundational economic income stablization and green incetives
direct_support_cols = [c for c in ["direct_payment_biss", "redistributive_support", "redistributive_support", "young_farmer_income", "eco_schemes"] if c in active_columns]

if direct_support_cols:
    direct_support_sum= sum(F.col(c) for c in direct_support_cols) # adding every direct_support_cols by row
    
    mart_direct_support= ireland24_master.select(required_base_inputs + direct_support_cols) \
        .withColumn("grand_total_eu_payout", F.col("total_eagf_income_support") + F.col("total_eafrd_income_support")) \
        .withColumn("grand_total_public_payout", F.col("grand_total_eu_payout") + F.col("national_co_funding")) \
        .select(mart_baseline + ["total_eagf_income_support"] + direct_support_cols) \
        .filter(direct_support_sum > 0.0) # eliminates empty row records

    

# Schema 2: Unified livestock and animal husbandry mart ( Pillar 1)
# Dedicated strictly to animal welfare and specific coupled livestock payouts
livestock_cols=  [c for c in ["livestock_welfare_and_support"] if c in active_columns]

if livestock_cols:
    livestock_sum= sum(F.col(c) for c in livestock_cols)
    mart_livestock_husbandry= ireland24_master.select(required_base_inputs + livestock_cols) \
        .withColumn("grand_total_eu_payout", F.col("total_eagf_income_support") + F.col("total_eafrd_income_support")) \
        .withColumn("grand_total_public_payout", F.col("grand_total_eu_payout") + F.col("national_co_funding")) \
        .select(mart_baseline + ["total_eagf_income_support"] + livestock_cols) \
        .filter(livestock_sum>0.0)
    


# Schema 3: Sectoral /Market interventions mart (Pillar 1)
# Focuses on supply-chain integrations, produer organisations,  and marketing aid
sectoral_market_cols = [c for c in ["school_nutrition_program", "producer_org_aid", "apiculture_support", "market_food_promotion"] if c in active_columns]
if sectoral_market_cols:
    sectoral_market_sum= sum(F.col(c) for c in sectoral_market_cols)
    mart_sectoral_markets= ireland24_master.select(required_base_inputs + sectoral_market_cols) \
        .withColumn("grand_total_eu_payout", F.col("total_eagf_income_support") + F.col("total_eafrd_income_support")) \
        .withColumn("grand_total_public_payout", F.col("grand_total_eu_payout") + F.col("national_co_funding")) \
        .select(mart_baseline + ["total_eagf_income_support"] + sectoral_market_cols) \
        .filter(sectoral_market_sum > 0.0)


# Schema 4:  Rural development & Environment Mart (Pillar 2)
# Tracks long-term infrastructure, environmental constraints, and organic transitions
# Note:  This  is the  only mart that includes national_co_funding and the EAFRD Pillar !

rural_dev_cols= [c for c in  [
    "agri_environmental_commitments", "natural_constraints_anc", "natura_2000_disadvantages", "physical_capital_investments"
, "knowledge_transfer_training", "rural_business_startup", "village_renewal_infra" ] if c in active_columns]

if rural_dev_cols:
    rural_dev_sum= sum(F.col(c) for c in rural_dev_cols)
    mart_rural_development = ireland24_master.select(required_base_inputs + rural_dev_cols) \
        .withColumn("grand_total_eu_payout", F.col("total_eagf_income_support") + F.col("total_eafrd_income_support")) \
        .withColumn("grand_total_public_payout", F.col("grand_total_eu_payout") + F.col("national_co_funding")) \
        .select(mart_baseline + ["national_co_funding", "total_eafrd_income_support"] + rural_dev_cols) \
        .filter(rural_dev_sum> 0.0)


In [15]:
mart_rural_development.printSchema()

root
 |-- parent_entity_vat: string (nullable = true)
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- grand_total_public_payout: double (nullable = true)
 |-- grand_total_eu_payout: double (nullable = true)
 |-- national_co_funding: double (nullable = true)
 |-- total_eafrd_income_support: double (nullable = true)
 |-- agri_environmental_commitments: double (nullable = false)
 |-- natural_constraints_anc: double (nullable = false)
 |-- natura_2000_disadvantages: double (nullable = false)
 |-- physical_capital_investments: double (nullable = false)
 |-- knowledge_transfer_training: double (nullable = false)
 |-- rural_business_startup: double (nullable = false)
 |-- village_renewal_infra: double (nullable = false)



In [ ]:
mart_rural_development.show(8, truncate=False)

In [16]:
mart_sectoral_markets.printSchema()

root
 |-- parent_entity_vat: string (nullable = true)
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- grand_total_public_payout: double (nullable = true)
 |-- grand_total_eu_payout: double (nullable = true)
 |-- total_eagf_income_support: double (nullable = true)
 |-- school_nutrition_program: double (nullable = false)
 |-- market_food_promotion: double (nullable = false)



In [ ]:
mart_sectoral_markets.show(8, truncate=False)

In [16]:
mart_livestock_husbandry.printSchema()

root
 |-- parent_entity_vat: string (nullable = true)
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- grand_total_public_payout: double (nullable = true)
 |-- grand_total_eu_payout: double (nullable = true)
 |-- total_eagf_income_support: double (nullable = true)
 |-- livestock_welfare_and_support: double (nullable = false)



In [20]:
mart_livestock_husbandry.show(8, truncate=False)

+------------------------+------------------------------------------------+------------+-------+----+-------------------------+---------------------+-------------------------+-----------------------------+
|parent_entity_vat       |Name of the beneficiary/Legal entity/association|municipality|country|year|grand_total_public_payout|grand_total_eu_payout|total_eagf_income_support|livestock_welfare_and_support|
+------------------------+------------------------------------------------+------------+-------+----+-------------------------+---------------------+-------------------------+-----------------------------+
|ANON_VAT_IRELAND_0000225|MARTIN & RYAN GIBBONS                           |GALWAY      |IRELAND|2024|3221.91                  |3221.91              |3221.91                  |0.0                          |
|ANON_VAT_IRELAND_000069a|MERVYN & DAWN BUTTIMER                          |CORK        |IRELAND|2024|12173.37                 |12173.37             |12173.37                 |0

In [ ]:
mart_livestock_husbandry.show.select(
    
).distinct().limit(6).s

In [18]:
mart_direct_support.printSchema()

root
 |-- parent_entity_vat: string (nullable = true)
 |-- Name of the beneficiary/Legal entity/association: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- country: string (nullable = true)
 |-- year: string (nullable = true)
 |-- grand_total_public_payout: double (nullable = true)
 |-- grand_total_eu_payout: double (nullable = true)
 |-- total_eagf_income_support: double (nullable = true)
 |-- direct_payment_biss: double (nullable = false)
 |-- redistributive_support: double (nullable = false)
 |-- redistributive_support: double (nullable = false)
 |-- young_farmer_income: double (nullable = false)
 |-- eco_schemes: double (nullable = false)



In [19]:
mart_direct_support.show(8, truncate=False)

+------------------------+------------------------------------------------+------------+-------+----+-------------------------+---------------------+-------------------------+-------------------+----------------------+----------------------+-------------------+-----------+
|parent_entity_vat       |Name of the beneficiary/Legal entity/association|municipality|country|year|grand_total_public_payout|grand_total_eu_payout|total_eagf_income_support|direct_payment_biss|redistributive_support|redistributive_support|young_farmer_income|eco_schemes|
+------------------------+------------------------------------------------+------------+-------+----+-------------------------+---------------------+-------------------------+-------------------+----------------------+----------------------+-------------------+-----------+
|ANON_VAT_IRELAND_0000225|MARTIN & RYAN GIBBONS                           |GALWAY      |IRELAND|2024|3221.91                  |3221.91              |3221.91                  |322

In [ ]:
spark.stop()